# 01 — Data Ingestion

This notebook demonstrates the data ingestion step of the Weather Intelligence Pipeline.

It collects historical weather data and 7-day forecast data from the Open-Meteo API, saves the raw outputs as parquet files, and loads them into DuckDB.

In the automated workflow, this ingestion logic is handled by `src/pipeline.py`.

## 1. Imports & Setup

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.config import (
    CITIES,
    START_DATE,
    END_DATE,
    DAILY_VARIABLES,
)

from src.ingestion import (
    fetch_all_cities,
    fetch_forecast_all_cities,
)

from src.db import (
    create_schemas,
    load_raw_data,
    run_query,
)

## 2. Configuration Check

In [2]:
print("Configured cities:")
for city in CITIES:
    print(f"- {city['name']} ({city['latitude']}, {city['longitude']})")

print("\nHistorical date range:")
print(f"{START_DATE} → {END_DATE}")

print("\nDaily weather variables:")
for variable in DAILY_VARIABLES:
    print(f"- {variable}")

Configured cities:
- Baku (40.41, 49.87)
- Lankaran (38.75, 48.85)
- Guba (41.36, 48.51)
- Gabala (40.58, 47.5)
- Shaki (41.11, 47.1)

Historical date range:
2020-04-27 → 2026-04-27

Daily weather variables:
- temperature_2m_max
- precipitation_sum
- wind_speed_10m_max
- relative_humidity_2m_mean
- cloud_cover_mean
- apparent_temperature_max
- sunshine_duration


## 3. Prepare Raw Data Folders

In [3]:
raw_historical_dir = PROJECT_ROOT / "data" / "raw" / "historical"
raw_forecast_dir = PROJECT_ROOT / "data" / "raw" / "forecast"

raw_historical_dir.mkdir(parents=True, exist_ok=True)
raw_forecast_dir.mkdir(parents=True, exist_ok=True)

for file in raw_historical_dir.glob("*.parquet"):
    file.unlink()

for file in raw_forecast_dir.glob("*.parquet"):
    file.unlink()

print("Raw historical folder:", raw_historical_dir)
print("Raw forecast folder:", raw_forecast_dir)
print("Old raw parquet files removed.")

Raw historical folder: C:\Users\viktus\Desktop\for_labs\module5\m5-project-weather-pipeline\data\raw\historical
Raw forecast folder: C:\Users\viktus\Desktop\for_labs\module5\m5-project-weather-pipeline\data\raw\forecast
Old raw parquet files removed.


## 4. Fetch Historical Weather Data


In [4]:
historical_data = fetch_all_cities(
    cities_config=CITIES,
    start_date=START_DATE,
    end_date=END_DATE,
    variables=DAILY_VARIABLES,
)

print("Historical data fetched for:")
for city_name, df_city in historical_data.items():
    print(f"- {city_name}: {len(df_city)} rows")

Fetching historical data for Baku...
Baku: 2192 historical rows
Fetching historical data for Lankaran...
Lankaran: 2192 historical rows
Fetching historical data for Guba...
Guba: 2192 historical rows
Fetching historical data for Gabala...
Gabala: 2192 historical rows
Fetching historical data for Shaki...
Shaki: 2192 historical rows
Historical data fetched for:
- Baku: 2192 rows
- Lankaran: 2192 rows
- Guba: 2192 rows
- Gabala: 2192 rows
- Shaki: 2192 rows


## 5. Inspect Historical Data


In [5]:
for city_name, df_city in historical_data.items():
    print(f"\nCity: {city_name}")
    print(f"Rows: {len(df_city)}")
    print(f"Date range: {df_city['time'].min()} → {df_city['time'].max()}")
    print("Columns:")
    print(df_city.columns.tolist())
    print("Missing values:")
    print(df_city.isna().sum())


City: Baku
Rows: 2192
Date range: 2020-04-27 00:00:00 → 2026-04-27 00:00:00
Columns:
['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']
Missing values:
time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidity_2m_mean    0
cloud_cover_mean             0
apparent_temperature_max     0
sunshine_duration            0
city                         0
dtype: int64

City: Lankaran
Rows: 2192
Date range: 2020-04-27 00:00:00 → 2026-04-27 00:00:00
Columns:
['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']
Missing values:
time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidit

## 6. Save Historical Data


In [6]:
for city_name, df_city in historical_data.items():
    safe_city_name = city_name.lower().replace(" ", "_")
    file_path = raw_historical_dir / f"{safe_city_name}_historical.parquet"

    df_city.to_parquet(file_path, index=False)

    print(f"Saved historical data: {file_path.name}")

Saved historical data: baku_historical.parquet
Saved historical data: lankaran_historical.parquet
Saved historical data: guba_historical.parquet
Saved historical data: gabala_historical.parquet
Saved historical data: shaki_historical.parquet


## 7. Fetch Forecast Weather Data


In [7]:
forecast_data = fetch_forecast_all_cities(
    cities_config=CITIES,
    variables=DAILY_VARIABLES,
)

print("Forecast data fetched for:")
for city_name, df_city in forecast_data.items():
    print(f"- {city_name}: {len(df_city)} rows")

Fetching forecast data for Baku...
Baku: 7 forecast rows
Fetching forecast data for Lankaran...
Lankaran: 7 forecast rows
Fetching forecast data for Guba...
Guba: 7 forecast rows
Fetching forecast data for Gabala...
Gabala: 7 forecast rows
Fetching forecast data for Shaki...
Shaki: 7 forecast rows
Forecast data fetched for:
- Baku: 7 rows
- Lankaran: 7 rows
- Guba: 7 rows
- Gabala: 7 rows
- Shaki: 7 rows


## 8. Inspect and Save Forecast Data


### 8.1 — inspect forecast

In [8]:
for city_name, df_city in forecast_data.items():
    print(f"\nCity: {city_name}")
    print(f"Rows: {len(df_city)}")
    print(f"Date range: {df_city['time'].min()} → {df_city['time'].max()}")
    print("Columns:")
    print(df_city.columns.tolist())
    print("Missing values:")
    print(df_city.isna().sum())


City: Baku
Rows: 7
Date range: 2026-04-28 00:00:00 → 2026-05-04 00:00:00
Columns:
['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']
Missing values:
time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidity_2m_mean    0
cloud_cover_mean             0
apparent_temperature_max     0
sunshine_duration            0
city                         0
dtype: int64

City: Lankaran
Rows: 7
Date range: 2026-04-28 00:00:00 → 2026-05-04 00:00:00
Columns:
['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']
Missing values:
time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidity_2m_m

### 8.2 — save forecast

In [9]:
for city_name, df_city in forecast_data.items():
    safe_city_name = city_name.lower().replace(" ", "_")
    file_path = raw_forecast_dir / f"{safe_city_name}_forecast.parquet"

    df_city.to_parquet(file_path, index=False)

    print(f"Saved forecast data: {file_path.name}")

Saved forecast data: baku_forecast.parquet
Saved forecast data: lankaran_forecast.parquet
Saved forecast data: guba_forecast.parquet
Saved forecast data: gabala_forecast.parquet
Saved forecast data: shaki_forecast.parquet


## 9. Combine and Sanity Check Raw Data


### 9.1 — combine

In [10]:
historical_all = (
    pd.concat(historical_data.values(), ignore_index=True)
    .sort_values(["city", "time"])
    .reset_index(drop=True)
)

forecast_all = (
    pd.concat(forecast_data.values(), ignore_index=True)
    .sort_values(["city", "time"])
    .reset_index(drop=True)
)

print("Historical shape:", historical_all.shape)
print("Forecast shape:", forecast_all.shape)

Historical shape: (10960, 9)
Forecast shape: (35, 9)


### 9.2 — sanity checks

In [11]:
print("Historical duplicate rows:", historical_all.duplicated().sum())
print("Forecast duplicate rows:", forecast_all.duplicated().sum())

print("\nHistorical date range:")
print(historical_all["time"].min(), "→", historical_all["time"].max())

print("\nForecast date range:")
print(forecast_all["time"].min(), "→", forecast_all["time"].max())

print("\nHistorical city counts:")
print(historical_all["city"].value_counts())

print("\nForecast city counts:")
print(forecast_all["city"].value_counts())

print("\nColumn consistency:")
print("Same columns:", set(historical_all.columns) == set(forecast_all.columns))

Historical duplicate rows: 0
Forecast duplicate rows: 0

Historical date range:
2020-04-27 00:00:00 → 2026-04-27 00:00:00

Forecast date range:
2026-04-28 00:00:00 → 2026-05-04 00:00:00

Historical city counts:
city
Baku        2192
Gabala      2192
Guba        2192
Lankaran    2192
Shaki       2192
Name: count, dtype: int64

Forecast city counts:
city
Baku        7
Gabala      7
Guba        7
Lankaran    7
Shaki       7
Name: count, dtype: int64

Column consistency:
Same columns: True


## 10. Load Raw Data into DuckDB


### 10.1 — load into DuckDB

In [12]:
create_schemas()
load_raw_data()

print("Raw data loaded into DuckDB.")

Raw data loaded into DuckDB.


### 10.2 — verify DuckDB tables

In [13]:
tables = run_query("""
SELECT table_schema, table_name
FROM information_schema.tables
ORDER BY table_schema, table_name
""")

display(tables)

,table_schema,table_name
0,analytics,final_28d_forecast
1,analytics,future_28d_forecast
2,analytics,model_features
3,raw,forecast
4,raw,historical


In [14]:
historical_summary = run_query("""
SELECT
    MIN(time) AS min_date,
    MAX(time) AS max_date,
    COUNT(*) AS rows
FROM raw.historical
""")

forecast_summary = run_query("""
SELECT
    MIN(time) AS min_date,
    MAX(time) AS max_date,
    COUNT(*) AS rows
FROM raw.forecast
""")

print("Historical table summary:")
display(historical_summary)

print("Forecast table summary:")
display(forecast_summary)

Historical table summary:


,min_date,max_date,rows
0,2020-04-27,2026-04-27,10960


Forecast table summary:


,min_date,max_date,rows
0,2026-04-28,2026-05-04,35


## 11. Ingestion Summary

The ingestion step successfully collected historical weather data and 7-day forecast data for all configured cities.

Historical data was saved to:

`data/raw/historical/`

Forecast data was saved to:

`data/raw/forecast/`

Both raw datasets were loaded into DuckDB as:

- `raw.historical`
- `raw.forecast`

The raw data layer is now ready for data quality checks, cleaning, and feature engineering.